# Kiểm thử Suy luận Mô hình Đã Huấn luyện kết hợp RAG (1-LoRA + RAG Inference)

Notebook này thực hiện việc **đánh giá suy luận (Inference Test)** cuối cùng của mô hình **Llama-3.1-8B (1-LoRA) + RAG**.

**Cấu hình Cache & Tránh rác ổ C:**
- Sử dụng tệp `.env` để cấu hình đường dẫn cache Hugging Face và Unsloth trên ổ `T:`. Toàn bộ dữ liệu mô hình tải về sẽ không lưu ở ổ C.

**Quy trình suy luận:**
1. Học sinh gửi Đề bài (Prompt) và Bài viết (Essay) lên hệ thống.
2. Hệ thống tìm kiếm tương đồng trên Vector DB ChromaDB để lấy **2 bài viết tham khảo** cùng chủ đề kèm điểm chuẩn.
3. Ghép các thông tin vào Prompt Template hệ thống.
4. Mô hình 1-LoRA đã được fine-tune thực hiện đọc hiểu, chấm điểm 4 tiêu chí và xuất ra chuỗi JSON duy nhất.
5. Backend thực hiện parse JSON để lấy điểm số và hiển thị biểu đồ nhận xét.

In [ ]:
import os
from dotenv import load_dotenv

# Nạp các biến môi trường cấu hình cache trước khi load Unsloth/Transformers
load_dotenv(os.path.abspath("../.env"))

import sys
import json
from unsloth import FastLanguageModel

# Thêm đường dẫn src/ để sử dụng rag_utils
sys.path.append(os.path.abspath("../src"))
from rag import rag_utils

## 1. Nạp Mô hình đã Fine-tuned & Kích hoạt chế độ suy luận nhanh

Chúng ta nạp trực tiếp mô hình nền lượng hóa kết hợp với LoRA Adapter đã lưu trong thư mục `adapters/`.

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

ADAPTER_DIR = "../adapters/llama_8b_1lora_aes"

print(f"Đang tải mô hình từ: {ADAPTER_DIR}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Kích hoạt chế độ suy luận nhanh của Unsloth (tăng tốc gấp 2 lần)
FastLanguageModel.for_inference(model)

## 2. Nạp Cơ sở dữ liệu Vector RAG

In [ ]:
VECTOR_DB_DIR = "../data/processed/chroma_db"
vectordb = rag_utils.load_vector_db(VECTOR_DB_DIR)
print("✔ Đã tải ChromaDB thành công.")

## 3. Xây dựng Pipeline chấm điểm IELTS hoàn chỉnh

Chúng ta tích hợp RAG và Model Generation vào một hàm duy nhất.

In [ ]:
IELTS_EVAL_PROMPT_TEMPLATE = """You are a highly experienced IELTS writing examiner. Your goal is to provide a precise and consistent evaluation of an essay by following a structured reasoning process.

**CONTEXT (Reference Essays with Scores):**
{context}

**NEW ESSAY TO GRADE:**
{question}

**EVALUATION PROCESS (Think step-by-step):**
1. Task Response (TR) Analysis: Assess how well the 'NEW ESSAY' addresses the prompt. Compare its quality to the TR scores in the 'CONTEXT'.
2. Coherence and Cohesion (CC) Analysis: Assess structure, paragraphing, and linking. Compare to CC scores in the 'CONTEXT'.
3. Lexical Resource (LR) Analysis: Assess range and accuracy of vocabulary. Compare to LR scores in the 'CONTEXT'.
4. Grammatical Range and Accuracy (GRA) Analysis: Assess grammar range and accuracy. Compare to GRA scores in the 'CONTEXT'.

**FINAL OUTPUT FORMAT (Strict JSON):**
Your entire response MUST be a single valid JSON object containing exactly these fields. Do NOT include markdown code blocks or explanations outside JSON.
{{
  "Task_Response": {{
    "Band": <score>,
    "Comment": "<brief justification>"
  }},
  "Coherence_and_Cohesion": {{
    "Band": <score>,
    "Comment": "<brief justification>"
  }},
  "Lexical_Resource": {{
    "Band": <score>,
    "Mistakes": ["<mistake1>", "<mistake2>"],
    "Corrections": ["<correction1>", "<correction2>"],
    "Comment": "<brief justification>"
  }},
  "Grammatical_Range_and_Accuracy": {{
    "Band": <score>,
    "Mistakes": ["<mistake1>", "<mistake2>"],
    "Corrections": ["<correction1>", "<correction2>"],
    "Comment": "<brief justification>"
  }},
  "General_Feedback": "<constructive feedback>"
}}

JSON Response:
"""

In [ ]:
def evaluate_essay(essay_prompt, essay_text):
    # 1. Truy xuất RAG
    retrieved_docs = rag_utils.retrieve_examples(vectordb, essay_text, k=2)
    context_str = rag_utils.format_rag_context(retrieved_docs)
    
    # 2. Xây dựng Prompt mẫu
    final_prompt = rag_utils.format_evaluation_prompt(
        prompt_template=IELTS_EVAL_PROMPT_TEMPLATE,
        context=context_str,
        essay_prompt=essay_prompt,
        essay_text=essay_text
    )
    
    # 3. Tokenize và Sinh phản hồi
    inputs = tokenizer([final_prompt], return_tensors="pt").to("cuda")
    
    print("⌛ Đang sinh kết quả chấm điểm từ Llama-3.1-8B (1-LoRA)...\n")
    outputs = model.generate(
        **inputs,
        max_new_tokens=768,
        use_cache=True,
        temperature=0.1,    # Nhiệt độ thấp giúp JSON sinh ra nhất quán và ổn định hơn
        top_p=0.9
    )
    
    # 4. Giải mã kết quả
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    response_str = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    return response_str

## 4. Chạy kiểm thử trên bài viết thực tế

Gửi một bài viết mẫu và thực hiện parse JSON kết quả.

In [ ]:
test_prompt = "Some people think that universities should provide graduates with the knowledge and skills needed in the workplace. Discuss both views."
test_essay = """
Nowadays, it is true that finding a job has become very competitive. Therefore, many people support the idea that universities should focus on vocational training. From my perspective, while providing professional skills is highly necessary, teaching theoretical knowledge for its own sake is also a fundamental responsibility of academic institutions.

On the one hand, specialized training directly prepares students for their future careers. For instance, engineering or nursing students cannot perform their duties without hands-on lab sessions and internship experiences. In today's market, employers value candidates who can contribute immediately without requiring expensive training programs. Thus, vocational modules help increase the employment rate among new graduates.

On the other hand, theoretical sciences foster critical thinking and problem-solving abilities. Pure academic disciplines like mathematics, philosophy, or history expand students' cognitive horizons, allowing them to comprehend complex global issues. If colleges only teach practical skills, they will become trade schools, and society might lose long-term scientific progress.

In conclusion, universities must strike a balance. They should offer courses that teach both workplace skills and theoretical academic knowledge to create well-rounded graduates.
"""

response = evaluate_essay(test_prompt, test_essay)
print("=== PHẢN HỒI THÔ TỪ MODEL ===")
print(response)
print("\n" + "="*60 + "\n")

# Thử nghiệm chuyển đổi sang đối tượng JSON trong Python
try:
    json_data = json.loads(response)
    print("✔ ĐÃ PARSE JSON THÀNH CÔNG:")
    print(json.dumps(json_data, indent=2))
except Exception as e:
    print("❌ LỖI PARSE JSON:", e)
    print("Mẹo khắc phục: Kiểm tra lại prompt hoặc hạ nhiệt độ sinh (temperature) xuống.")